# 02. Training Neural Network

Notebook ini melatih dua varian Neural Network berbasis PyTorch pada dataset MNIST dan Fashion MNIST.

Arsitektur yang digunakan:

1. NN-1: 784 → 128 → 10
2. NN-2: 784 → 256 → Dropout(0.3) → 128 → 10

Hasil evaluasi disimpan ke `results/nn_results.csv`, sedangkan model terbaik untuk setiap dataset disimpan ke folder `models/`.

In [1]:
import os
import sys
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
def get_project_root():
    cwd = Path.cwd().resolve()
    if cwd.name.lower() == "notebooks":
        return cwd.parent
    return cwd

PROJECT_ROOT = get_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
CM_DIR = RESULTS_DIR / "confusion_matrix"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [DATA_DIR, RESULTS_DIR, CM_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\UTS_Data_Mining


In [3]:
CONFIG = {
    "random_seed": 42,
    "batch_size": 64,
    "epochs": 10,
    "learning_rate": 0.001,
    "num_classes": 10,
    "input_size": 784,
    "test_subset": 2000
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["random_seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device digunakan:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA tidak tersedia. Training tetap berjalan menggunakan CPU.")

Device digunakan: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
transform = transforms.ToTensor()

dataset_objects = {
    "MNIST": {
        "train": datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform),
        "test": datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform),
        "model_file": "best_nn_mnist.pth"
    },
    "FashionMNIST": {
        "train": datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform),
        "test": datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform),
        "model_file": "best_nn_fashion_mnist.pth"
    }
}

for name, obj in dataset_objects.items():
    print(f"{name}: train={len(obj['train'])}, test={len(obj['test'])}")

MNIST: train=60000, test=10000
FashionMNIST: train=60000, test=10000


In [5]:
def fixed_subset(dataset, subset_size, seed=42):
    subset_size = min(subset_size, len(dataset))
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(dataset), size=subset_size, replace=False)
    return Subset(dataset, indices)

def create_dataloaders(train_dataset, test_dataset):
    test_subset = fixed_subset(
        test_dataset,
        subset_size=CONFIG["test_subset"],
        seed=CONFIG["random_seed"]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=0
    )

    test_loader = DataLoader(
        test_subset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=0
    )

    return train_loader, test_loader

In [6]:
class NN1_SimpleMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


class NN2_DeeperMLP(nn.Module):
    def __init__(self, input_size=784, num_classes=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


model_configs = {
    "NN-1": {
        "class": NN1_SimpleMLP,
        "configuration": "784-128-10, ReLU"
    },
    "NN-2": {
        "class": NN2_DeeperMLP,
        "configuration": "784-256-Dropout(0.3)-128-10, ReLU"
    }
}

In [7]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def predict_nn(model, dataloader, device):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.numpy())
            y_pred.extend(predicted.cpu().numpy())

    return np.array(y_true), np.array(y_pred)


def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

In [8]:
results = []

for dataset_name, dataset_data in dataset_objects.items():
    print("=" * 80)
    print(f"Dataset: {dataset_name}")

    train_loader, test_loader = create_dataloaders(
        dataset_data["train"],
        dataset_data["test"]
    )

    best_f1 = -1.0
    best_model_state = None
    best_model_name = None

    for model_name, config in model_configs.items():
        print("-" * 80)
        print(f"Training {model_name} pada {dataset_name}")

        set_seed(CONFIG["random_seed"])

        model = config["class"](
            input_size=CONFIG["input_size"],
            num_classes=CONFIG["num_classes"]
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(
            model.parameters(),
            lr=CONFIG["learning_rate"]
        )

        start_time = time.time()

        for epoch in range(CONFIG["epochs"]):
            train_loss, train_acc = train_one_epoch(
                model=model,
                dataloader=train_loader,
                criterion=criterion,
                optimizer=optimizer,
                device=device
            )

            print(
                f"Epoch {epoch+1:02d}/{CONFIG['epochs']} | "
                f"Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.4f}"
            )

        training_time = time.time() - start_time

        y_true, y_pred = predict_nn(model, test_loader, device)
        metrics = calculate_metrics(y_true, y_pred)

        row = {
            "dataset": dataset_name,
            "model_group": "Neural Network",
            "model": model_name,
            "configuration": config["configuration"],
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "training_time_seconds": training_time,
            "test_size": len(y_true)
        }

        results.append(row)

        print("Hasil evaluasi:")
        print(row)

        if metrics["f1_score"] > best_f1:
            best_f1 = metrics["f1_score"]
            best_model_state = model.state_dict()
            best_model_name = model_name

    model_path = MODELS_DIR / dataset_data["model_file"]
    torch.save({
        "dataset": dataset_name,
        "model_name": best_model_name,
        "model_state_dict": best_model_state,
        "best_f1_score": best_f1,
        "config": CONFIG
    }, model_path)

    print(f"Model NN terbaik untuk {dataset_name}: {best_model_name}")
    print(f"Disimpan ke: {model_path}")

Dataset: MNIST
--------------------------------------------------------------------------------
Training NN-1 pada MNIST
Epoch 01/10 | Loss: 0.3448 | Train Acc: 0.9062
Epoch 02/10 | Loss: 0.1580 | Train Acc: 0.9536
Epoch 03/10 | Loss: 0.1083 | Train Acc: 0.9682
Epoch 04/10 | Loss: 0.0812 | Train Acc: 0.9760
Epoch 05/10 | Loss: 0.0640 | Train Acc: 0.9808
Epoch 06/10 | Loss: 0.0513 | Train Acc: 0.9849
Epoch 07/10 | Loss: 0.0413 | Train Acc: 0.9877
Epoch 08/10 | Loss: 0.0336 | Train Acc: 0.9903
Epoch 09/10 | Loss: 0.0273 | Train Acc: 0.9924
Epoch 10/10 | Loss: 0.0227 | Train Acc: 0.9935
Hasil evaluasi:
{'dataset': 'MNIST', 'model_group': 'Neural Network', 'model': 'NN-1', 'configuration': '784-128-10, ReLU', 'accuracy': 0.9755, 'precision': 0.975307611590463, 'recall': 0.9757635164099158, 'f1_score': 0.9753806837178487, 'training_time_seconds': 78.21813583374023, 'test_size': 2000}
--------------------------------------------------------------------------------
Training NN-2 pada MNIST
Ep

In [9]:
nn_results_df = pd.DataFrame(results)
nn_results_df = nn_results_df.sort_values(
    by=["dataset", "f1_score"],
    ascending=[True, False]
).reset_index(drop=True)

output_path = RESULTS_DIR / "nn_results.csv"
nn_results_df.to_csv(output_path, index=False)

display(nn_results_df)
print("Hasil Neural Network disimpan ke:", output_path)

,dataset,model_group,model,configuration,accuracy,precision,recall,f1_score,training_time_seconds,test_size
0,FashionMNIST,Neural Network,NN-1,"784-128-10, ReLU",0.8810,0.883069,0.881996,0.880162,80.857809,2000
1,FashionMNIST,Neural Network,NN-2,"784-256-Dropout(0.3)-128-10, ReLU",0.8745,0.878529,0.876508,0.873482,93.764499,2000
2,MNIST,Neural Network,NN-2,"784-256-Dropout(0.3)-128-10, ReLU",0.9810,0.980873,0.980821,0.980833,93.985983,2000
3,MNIST,Neural Network,NN-1,"784-128-10, ReLU",0.9755,0.975308,0.975764,0.975381,78.218136,2000


Hasil Neural Network disimpan ke: C:\UTS_Data_Mining\results\nn_results.csv


## Catatan

File `nn_results.csv` akan digunakan kembali pada notebook `04_evaluation_and_comparison.ipynb` untuk digabungkan dengan hasil SVM.